# 🖼️ CNN — Quatre Projets de Classification d'Images

Ce notebook couvre **4 exercices pratiques** sur les réseaux de neurones convolutifs (CNN) :

| # | Exercice | Concept clé |
|---|----------|-------------|
| 1 | Classification d'objets (CIFAR-10) | Architecture CNN + Data Augmentation |
| 2 | Classification avec peu de données | Anti-overfitting : Dropout, Early Stopping |
| 3 | Transfer Learning (ResNet) | Feature Extraction vs Fine-tuning |
| 4 | Visualisation des filtres CNN | Interprétabilité du modèle |

> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

In [ ]:
# ── Installation et imports globaux ──────────────────────────────────────────
!pip install -q torch torchvision matplotlib seaborn scikit-learn

import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import confusion_matrix, classification_report

sns.set_theme(style='darkgrid')
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Environnement prêt — Appareil : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

---
# 🏷️ Exercice 1 — Classification d'Objets du Quotidien (CIFAR-10)

## Objectif
Construire un CNN pour classifier **10 catégories d'objets** et mesurer l'impact de la **data augmentation** sur la robustesse du modèle.

### Dataset : CIFAR-10
- **60 000 images** couleur de 32×32 pixels
- **10 classes** : avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion
- Divisé en 50 000 train + 10 000 test

### Data Augmentation
Technique qui crée des variantes artificielles de chaque image pendant l'entraînement (retournement, recadrage, changement de couleur…). Le modèle apprend ainsi à être **invariant** à ces transformations.

In [ ]:
# ── Transformations : avec et sans augmentation ───────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)  # Moyennes RGB du dataset CIFAR-10
CIFAR_STD  = (0.2470, 0.2435, 0.2616)  # Écarts-types RGB

# ── Sans augmentation : uniquement normalisation ──────────────────────────────
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# ── Avec augmentation ─────────────────────────────────────────────────────────
# RandomHorizontalFlip : retournement horizontal aléatoire (50% de chance)
# RandomCrop          : découpe aléatoire après padding de 4px (simule différents cadrage)
# ColorJitter         : légères variations de luminosité/saturation (simule différents éclairages)
transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

transform_test = transform_basic  # Le test ne doit JAMAIS être augmenté

# ── Chargement des datasets ───────────────────────────────────────────────────
train_basic = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=transform_basic)
train_augm  = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=transform_augmented)
test_cifar  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_test)

loader_basic = DataLoader(train_basic, batch_size=128, shuffle=True,  num_workers=2)
loader_augm  = DataLoader(train_augm,  batch_size=128, shuffle=True,  num_workers=2)
loader_test  = DataLoader(test_cifar,  batch_size=128, shuffle=False, num_workers=2)

CIFAR_CLASSES = ('Avion','Voiture','Oiseau','Chat','Cerf','Chien','Grenouille','Cheval','Bateau','Camion')
print(f"✅ CIFAR-10 chargé : {len(train_basic)} train | {len(test_cifar)} test")

In [ ]:
# ── Visualisation de l'effet de l'augmentation ────────────────────────────────
# Affichons la même image avec et sans augmentation

def denorm(tensor, mean=CIFAR_MEAN, std=CIFAR_STD):
    """Inverse la normalisation pour afficher l'image."""
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i] = t[i] * s + m
    return torch.clamp(t, 0, 1)

# Récupérer la même image originale et ses versions augmentées
orig_img, label = torchvision.datasets.CIFAR10('./data', train=True, download=False,
                  transform=transforms.ToTensor())[0]

aug_transform_only = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

fig, axes = plt.subplots(1, 7, figsize=(14, 3))
axes[0].imshow(orig_img.permute(1, 2, 0))
axes[0].set_title('Original', fontsize=9, fontweight='bold')
axes[0].axis('off')

pil_img = transforms.ToPILImage()(orig_img)
for i in range(1, 7):
    aug = transforms.ToTensor()(aug_transform_only(pil_img))
    axes[i].imshow(aug.permute(1, 2, 0))
    axes[i].set_title(f'Augm. {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle(f'Data Augmentation — classe : {CIFAR_CLASSES[label]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Chaque epoch, le modèle voit des variantes différentes → apprentissage plus robuste.")

In [ ]:
# ── Architecture CNN ──────────────────────────────────────────────────────────
# Structure classique : blocs [Conv → BN → ReLU → MaxPool] empilés
# + tête de classification Dense

class CIFAR_CNN(nn.Module):
    """
    CNN pour CIFAR-10 (images 32×32×3).

    Bloc convolutif standard :
      Conv2d        → extrait des features locales (bords, textures, formes)
      BatchNorm2d   → stabilise l'entraînement (normalise les activations)
      ReLU          → non-linéarité (évite que le réseau reste linéaire)
      MaxPool2d(2)  → divise la résolution par 2 (résumé spatial)
    """
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            # Bloc 1 : 32×32×3  → 32×32×64 → 16×16×64
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # → 16×16
            nn.Dropout2d(0.1),

            # Bloc 2 : 16×16×64 → 16×16×128 → 8×8×128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # → 8×8
            nn.Dropout2d(0.1),

            # Bloc 3 : 8×8×128 → 8×8×256 → 4×4×256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # → 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 128),          nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model_test = CIFAR_CNN()
dummy = torch.zeros(1, 3, 32, 32)
print(f"✅ Architecture validée — sortie : {model_test(dummy).shape}")
print(f"   Paramètres totaux : {sum(p.numel() for p in model_test.parameters()):,}")

In [ ]:
# ── Fonction d'entraînement générique ────────────────────────────────────────
def train_cnn(model, train_loader, test_loader, epochs=20, lr=1e-3, label=''):
    """Entraîne un CNN et retourne l'historique train/test accuracy."""
    model = model.to(DEVICE)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'test_acc': [], 'train_loss': []}

    for epoch in range(epochs):
        # ── Phase entraînement ────────────────────────────────────────────────
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += out.argmax(1).eq(y).sum().item()
            total   += y.size(0)
        scheduler.step()
        train_acc = 100 * correct / total

        # ── Phase évaluation ──────────────────────────────────────────────────
        model.eval()
        c, t = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                c += model(X).argmax(1).eq(y).sum().item()
                t += y.size(0)
        test_acc = 100 * c / t

        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['train_loss'].append(running_loss / len(train_loader))

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{label}] Epoch {epoch+1:>2}/{epochs} | "
                  f"Train: {train_acc:.1f}% | Test: {test_acc:.1f}%")

    return model, history


print("🔵 Entraînement SANS augmentation...")
model_basic, hist_basic = train_cnn(
    CIFAR_CNN(), loader_basic, loader_test, epochs=20, label='Sans augm')

print("\n🟢 Entraînement AVEC augmentation...")
model_augm, hist_augm = train_cnn(
    CIFAR_CNN(), loader_augm, loader_test, epochs=20, label='Avec augm')

In [ ]:
# ── Comparaison et matrice de confusion ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, 21)

for ax, key, title in zip(axes, ['train_acc', 'test_acc'], ['Train Accuracy', 'Test Accuracy']):
    ax.plot(ep, hist_basic[key], 'o-', color='tomato',    linewidth=2, label='Sans augmentation', markersize=4)
    ax.plot(ep, hist_augm[key],  's-', color='royalblue', linewidth=2, label='Avec augmentation', markersize=4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend()

plt.suptitle('Exercice 1 — Impact de la Data Augmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Matrice de confusion (modèle avec augmentation) ───────────────────────────
model_augm.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in loader_test:
        preds = model_augm(X.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR_CLASSES, yticklabels=CIFAR_CLASSES, ax=ax)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title('Matrice de confusion — Modèle avec augmentation', fontsize=12)
plt.tight_layout()
plt.show()

final_basic = hist_basic['test_acc'][-1]
final_augm  = hist_augm['test_acc'][-1]
print(f"📊 Accuracy finale — Sans augm: {final_basic:.1f}% | Avec augm: {final_augm:.1f}%")
print(f"   Gain : +{final_augm - final_basic:.1f} points")

---
# 📉 Exercice 2 — Classification avec Peu de Données

## Objectif
Entraîner un CNN sur **seulement 500 images** (10% de CIFAR-10) et comparer l'efficacité de 4 techniques anti-overfitting.

### Le problème de l'overfitting
Avec peu de données, le modèle **mémorise** les exemples d'entraînement au lieu d'apprendre des patterns généraux. Il devient très précis sur le train mais échoue sur le test.

```
Overfitting → train_acc ↑↑↑  mais  test_acc ↓ (écart croissant)
```

### Techniques testées :
| Technique | Principe |
|-----------|----------|
| **Baseline** | Aucune protection → référence |
| **Dropout** | Désactive aléatoirement des neurones → force la redondance |
| **Data Augmentation** | Crée de nouvelles variantes → augmente la diversité effective |
| **Early Stopping** | Arrête dès que la val_loss stagne → évite la sur-spécialisation |
| **Tout combiné** | Les 3 ensemble → résultat optimal |

In [ ]:
# ── Création d'un petit dataset (500 images) ──────────────────────────────────
# On sous-échantillonne CIFAR-10 pour simuler un dataset de taille limitée
# 50 images par classe × 10 classes = 500 images

N_PER_CLASS = 50
full_train  = torchvision.datasets.CIFAR10('./data', train=True, download=False,
                                            transform=transform_basic)

# Sélectionner N_PER_CLASS exemples par classe
indices_small = []
class_counts  = {i: 0 for i in range(10)}
for idx, (_, label) in enumerate(full_train):
    if class_counts[label] < N_PER_CLASS:
        indices_small.append(idx)
        class_counts[label] += 1
    if all(v == N_PER_CLASS for v in class_counts.values()):
        break

small_dataset = Subset(full_train, indices_small)
# Validation set : 20% du petit dataset
n_val = len(small_dataset) // 5
n_tr  = len(small_dataset) - n_val
small_train, small_val = random_split(small_dataset, [n_tr, n_val],
                                      generator=torch.Generator().manual_seed(42))

loader_small_tr  = DataLoader(small_train, batch_size=32, shuffle=True)
loader_small_val = DataLoader(small_val,   batch_size=32, shuffle=False)

print(f"✅ Petit dataset : {n_tr} train | {n_val} val | {len(test_cifar)} test")
print(f"   (vs {len(full_train)} images dans le dataset complet)")

In [ ]:
# ── Modèles pour l'exercice 2 ─────────────────────────────────────────────────

class SmallCNN(nn.Module):
    """
    Architecture légère (peu de paramètres) pour éviter le surapprentissage.
    Moins de canaux = moins de capacité de mémorisation.
    """
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16×16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8×8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),           # Dropout configurable
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Dropout(dropout_rate / 2),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class EarlyStopping:
    """
    Arrête l'entraînement si la validation loss ne s'améliore plus.
    Sauvegarde les meilleurs poids pour les restaurer après l'arrêt.
    """
    def __init__(self, patience=10, delta=0.001):
        self.patience   = patience
        self.delta      = delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.best_weights = None
        self.stopped    = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss    = val_loss
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter      = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped


def run_experiment(config_name, dropout=0.0, augment=False, early_stop=False, epochs=50):
    """
    Entraîne un SmallCNN avec la configuration spécifiée.
    Retourne l'historique et l'accuracy finale sur le test.
    """
    # Transform selon augmentation ou non
    tr_transform = transform_augmented if augment else transform_basic
    full_aug = torchvision.datasets.CIFAR10('./data', train=True, download=False, transform=tr_transform)
    small_aug = Subset(full_aug, indices_small)
    n_val_a = len(small_aug) // 5
    small_tr_a, small_val_a = random_split(small_aug, [len(small_aug) - n_val_a, n_val_a],
                                           generator=torch.Generator().manual_seed(42))
    tr_loader = DataLoader(small_tr_a,  batch_size=32, shuffle=True)
    va_loader = DataLoader(small_val_a, batch_size=32, shuffle=False)

    model = SmallCNN(dropout_rate=dropout).to(DEVICE)
    opt   = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit  = nn.CrossEntropyLoss()
    es    = EarlyStopping(patience=12) if early_stop else None

    hist = {'train_acc': [], 'val_acc': [], 'val_loss': []}
    stopped_at = epochs

    for epoch in range(epochs):
        model.train()
        c, t = 0, 0
        for X, y in tr_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X), y); loss.backward(); opt.step()
            c += model(X).argmax(1).eq(y).sum().item(); t += y.size(0)
        hist['train_acc'].append(100 * c / t)

        model.eval()
        vc, vt, vl = 0, 0, 0.0
        with torch.no_grad():
            for X, y in va_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out = model(X); vl += crit(out, y).item() * y.size(0)
                vc += out.argmax(1).eq(y).sum().item(); vt += y.size(0)
        hist['val_acc'].append(100 * vc / vt)
        hist['val_loss'].append(vl / vt)

        if es and es(hist['val_loss'][-1], model):
            stopped_at = epoch + 1
            model.load_state_dict(es.best_weights)
            print(f"  [{config_name}] ⏹️  Early stopping à l'epoch {stopped_at}")
            break

    # Test accuracy
    model.eval()
    tc, tt = 0, 0
    with torch.no_grad():
        for X, y in loader_test:
            X, y = X.to(DEVICE), y.to(DEVICE)
            tc += model(X).argmax(1).eq(y).sum().item(); tt += y.size(0)
    test_acc = 100 * tc / tt
    print(f"  [{config_name}] Test accuracy : {test_acc:.1f}% (arrêt epoch {stopped_at})")
    return hist, test_acc


# ── Lancement des 5 configurations ───────────────────────────────────────────
results_ex2 = {}
configs = [
    ('Baseline',          dict(dropout=0.0, augment=False, early_stop=False)),
    ('Dropout(0.5)',      dict(dropout=0.5, augment=False, early_stop=False)),
    ('Augmentation',      dict(dropout=0.0, augment=True,  early_stop=False)),
    ('Early Stopping',    dict(dropout=0.0, augment=False, early_stop=True )),
    ('Tout combiné',      dict(dropout=0.4, augment=True,  early_stop=True )),
]

for name, kwargs in configs:
    print(f"\n🔧 {name}...")
    hist, tacc = run_experiment(name, **kwargs)
    results_ex2[name] = {'hist': hist, 'test_acc': tacc}

In [ ]:
# ── Visualisation : overfitting et comparaison ────────────────────────────────
colors = ['tomato', 'orange', 'steelblue', 'mediumseagreen', 'royalblue']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for (name, res), color in zip(results_ex2.items(), colors):
    h = res['hist']
    ep = range(1, len(h['train_acc']) + 1)
    axes[0].plot(ep, h['train_acc'], '--', color=color, alpha=0.5, linewidth=1)
    axes[0].plot(ep, h['val_acc'],   '-',  color=color, linewidth=2, label=name)

axes[0].set_title('Train (pointillés) vs Validation (plein)', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=9)

names  = list(results_ex2.keys())
taccs  = [v['test_acc'] for v in results_ex2.values()]
bars = axes[1].bar(names, taccs, color=colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, taccs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[1].set_title('Test Accuracy par configuration', fontsize=12)
axes[1].set_ylabel('Accuracy (%)')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Exercice 2 — Techniques Anti-Overfitting (500 images)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🔄 Exercice 3 — Transfer Learning avec ResNet

## Objectif
Utiliser un modèle **pré-entraîné sur ImageNet** (1.2M images, 1000 classes) et l'adapter à CIFAR-10.

### Deux approches :

| Approche | Description | Avantage |
|----------|-------------|----------|
| **Feature Extraction** | Geler les couches convolutives, entraîner seulement la tête | Très rapide, peu de données nécessaires |
| **Fine-tuning** | Dégeler toutes les couches, entraîner avec un LR faible | Meilleures performances, plus long |

### Pourquoi ça marche ?
Les premières couches d'un CNN apprennent des features **universelles** (bords, textures, couleurs) qui sont utiles pour n'importe quel dataset d'images. Seules les dernières couches sont spécifiques à la tâche.

In [ ]:
# ── Chargement du ResNet18 pré-entraîné ───────────────────────────────────────
# ResNet18 a été entraîné sur ImageNet (1000 classes, images 224×224)
# On doit l'adapter à CIFAR-10 (10 classes, images 32×32)

from torchvision.models import resnet18, ResNet18_Weights

# ── Approche 1 : Feature Extraction ──────────────────────────────────────────
# On gèle TOUS les paramètres pré-entraînés
# Seule la tête (dernière couche Dense) sera entraînée

def build_feature_extractor():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    # Geler tous les paramètres (requires_grad=False = pas de gradient = pas de mise à jour)
    for param in model.parameters():
        param.requires_grad = False

    # Adapter le premier layer pour CIFAR-10 (images 32×32 au lieu de 224×224)
    model.conv1    = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool  = nn.Identity()  # Supprimer le pooling initial (trop agressif pour 32×32)

    # Remplacer la tête par une nouvelle (entraînable) pour 10 classes
    in_features = model.fc.in_features  # 512 pour ResNet18
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 10)
    )
    # Dégeler la nouvelle tête + les adaptations
    for param in model.fc.parameters():
        param.requires_grad = True
    for param in model.conv1.parameters():
        param.requires_grad = True

    return model


# ── Approche 2 : Fine-tuning ──────────────────────────────────────────────────
def build_finetuned():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    in_features   = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 10)
    )
    # Tous les paramètres sont entraînables (fine-tuning complet)
    return model


fe_model = build_feature_extractor()
ft_model = build_finetuned()

# Compter les paramètres entraînables
def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"✅ Paramètres entraînables :")
print(f"   Feature Extraction : {count_trainable(fe_model):>9,} / {sum(p.numel() for p in fe_model.parameters()):,}")
print(f"   Fine-tuning        : {count_trainable(ft_model):>9,} / {sum(p.numel() for p in ft_model.parameters()):,}")

In [ ]:
# ── Entraînement des trois approches ─────────────────────────────────────────
# Pour le fine-tuning : LR plus faible pour les couches pré-entraînées
# (on ne veut pas les déstabiliser, juste les ajuster légèrement)

def train_transfer(model, label, lr_head=1e-3, lr_backbone=1e-4, epochs=15):
    model = model.to(DEVICE)

    # LR différentiels : plus faible pour le backbone pré-entraîné
    backbone_params = [p for name, p in model.named_parameters()
                       if 'fc' not in name and p.requires_grad]
    head_params     = [p for name, p in model.named_parameters() if 'fc' in name]

    optimizer = Adam([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': lr_head}
    ], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    hist = {'train_acc': [], 'test_acc': []}
    t_start = time.time()

    for epoch in range(epochs):
        model.train()
        c, t = 0, 0
        for X, y in loader_augm:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y); loss.backward(); optimizer.step()
            c += model(X).detach().argmax(1).eq(y).sum().item(); t += y.size(0)
        scheduler.step()
        hist['train_acc'].append(100 * c / t)

        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for X, y in loader_test:
                X, y = X.to(DEVICE), y.to(DEVICE)
                vc += model(X).argmax(1).eq(y).sum().item(); vt += y.size(0)
        hist['test_acc'].append(100 * vc / vt)

        if (epoch + 1) % 5 == 0:
            print(f"  [{label}] Epoch {epoch+1:>2} | Train: {hist['train_acc'][-1]:.1f}% | Test: {hist['test_acc'][-1]:.1f}%")

    elapsed = time.time() - t_start
    print(f"  [{label}] ⏱️  Temps : {elapsed:.0f}s | Meilleur test : {max(hist['test_acc']):.1f}%")
    return hist, elapsed


print("🔵 CNN from scratch (référence, 20 epochs)...")
_, hist_scratch = train_cnn(CIFAR_CNN(), loader_augm, loader_test, epochs=15, label='Scratch')
time_scratch = 0  # Déjà mesuré approximativement

print("\n🟡 Feature Extraction (ResNet18 gelé)...")
hist_fe, time_fe = train_transfer(fe_model, 'Feature Extraction', lr_backbone=0, epochs=15)

print("\n🟢 Fine-tuning (ResNet18 complet)...")
hist_ft, time_ft = train_transfer(ft_model, 'Fine-tuning', epochs=15)

In [ ]:
# ── Comparaison des 3 approches ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, 16)

axes[0].plot(ep, hist_scratch['test_acc'], 'o-', color='tomato',       lw=2, label='CNN from scratch')
axes[0].plot(ep, hist_fe['test_acc'],      's-', color='darkorange',   lw=2, label='Feature Extraction')
axes[0].plot(ep, hist_ft['test_acc'],      '^-', color='royalblue',    lw=2, label='Fine-tuning')
axes[0].set_title('Test Accuracy — 3 approches comparées', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()

# Tableau de synthèse
methods  = ['CNN Scratch', 'Feature Extr.', 'Fine-tuning']
best_acc = [max(hist_scratch['test_acc']), max(hist_fe['test_acc']), max(hist_ft['test_acc'])]
axes[1].barh(methods, best_acc, color=['tomato', 'darkorange', 'royalblue'], edgecolor='white')
for i, v in enumerate(best_acc):
    axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center', fontweight='bold')
axes[1].set_title('Meilleure accuracy par approche', fontsize=12)
axes[1].set_xlabel('Test Accuracy (%)')
axes[1].set_xlim(0, 100)

plt.suptitle('Exercice 3 — Transfer Learning vs From Scratch', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observations :")
print(f"   Feature Extraction atteint {max(hist_fe['test_acc']):.1f}% dès les premières epochs (convergence rapide)")
print(f"   Fine-tuning atteint  {max(hist_ft['test_acc']):.1f}% avec plus de flexibilité")
print(f"   CNN scratch atteint  {max(hist_scratch['test_acc']):.1f}% mais nécessite plus d'epochs")

---
# 🔍 Exercice 4 — Visualisation des Filtres CNN

## Objectif
Comprendre **ce qu'un CNN apprend** en visualisant ses filtres et les **feature maps** qu'ils produisent.

### Deux niveaux de visualisation :

1. **Les filtres eux-mêmes** : matrices de poids apprises dans la première couche Conv.  
   → Révèlent les patterns primitifs détectés : bords horizontaux, verticaux, diagonaux, fréquences...

2. **Les feature maps (activation maps)** : ce que le filtre voit quand il passe sur une image réelle.  
   → Montre les régions de l'image qui activent chaque filtre.

### Pourquoi c'est important ?
La visualisation est un outil d'**interprétabilité** : elle permet de déboguer le modèle, vérifier qu'il apprend des features pertinentes, et détecter des biais potentiels.

In [ ]:
# ── Visualisation des filtres de la 1ère couche convolutive ───────────────────
# On utilise le modèle entraîné dans l'exercice 1 (avec augmentation)

model_augm.eval()

# Récupérer les poids de la première couche Conv2d
# Shape : (n_filtres, 3, kernel_h, kernel_w) = (64, 3, 3, 3)
first_conv = None
for module in model_augm.modules():
    if isinstance(module, nn.Conv2d):
        first_conv = module
        break

filters = first_conv.weight.detach().cpu()
print(f"📐 Forme des filtres de la 1ère couche : {filters.shape}")
print(f"   ({filters.shape[0]} filtres × {filters.shape[1]} canaux × {filters.shape[2]}×{filters.shape[3]} pixels)")

# Normaliser les filtres pour l'affichage (valeurs entre 0 et 1)
def normalize_filter(f):
    f = f - f.min()
    return f / (f.max() + 1e-8)

# Afficher les 32 premiers filtres
n_show = 32
n_cols = 8
n_rows = n_show // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i >= n_show:
        ax.axis('off')
        continue
    filt = normalize_filter(filters[i]).permute(1, 2, 0).numpy()
    ax.imshow(filt)
    ax.set_title(f'F{i+1}', fontsize=7)
    ax.axis('off')

plt.suptitle('Exercice 4 — Filtres appris dans la 1ère couche convolutive', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Chaque filtre est une matrice 3×3 en RGB. On voit des patterns de couleur et d'orientation.")

In [ ]:
# ── Visualisation des Feature Maps (Activation Maps) ─────────────────────────
# On passe une image réelle dans le réseau et on regarde ce que
# chaque filtre de la 1ère couche produit comme activation.

# Récupérer une image de test
sample_img, sample_label = test_cifar[42]  # Choisir un exemple
img_tensor = sample_img.unsqueeze(0).to(DEVICE)  # Ajouter dimension batch

# Passer l'image dans la 1ère couche seulement
with torch.no_grad():
    activation = first_conv(img_tensor)  # Shape : (1, 64, 32, 32)
    activation = F.relu(activation)      # Appliquer ReLU (comme dans le vrai réseau)

activation = activation.squeeze(0).cpu()  # (64, 32, 32)

# Affichage : image originale + 16 feature maps
fig = plt.figure(figsize=(16, 7))
gs  = gridspec.GridSpec(3, 9, figure=fig)

# Image originale (dé-normalisée)
ax_orig = fig.add_subplot(gs[:, 0])
img_display = denorm(sample_img).permute(1, 2, 0).numpy()
ax_orig.imshow(img_display)
ax_orig.set_title(f'Original\n{CIFAR_CLASSES[sample_label]}', fontsize=10, fontweight='bold')
ax_orig.axis('off')

# Feature maps (3 lignes × 8 colonnes = 24 filtres)
for i in range(24):
    row, col = i // 8, i % 8
    ax = fig.add_subplot(gs[row, col + 1])
    fmap = activation[i].numpy()
    ax.imshow(fmap, cmap='viridis')
    ax.set_title(f'F{i+1}', fontsize=7)
    ax.axis('off')

plt.suptitle('Feature Maps — Activations de la 1ère couche Conv', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparaison : filtres couche 1 vs couche 3 ───────────────────────────────
# Les premières couches détectent des features simples (bords, couleurs)
# Les couches profondes détectent des features complexes (textures, formes)

# Extraire les activations à différentes profondeurs via hooks
activations_by_layer = {}

def make_hook(name):
    def hook(module, input, output):
        activations_by_layer[name] = output.detach()
    return hook

# Enregistrer des hooks sur les couches clés
conv_layers = [(name, m) for name, m in model_augm.named_modules() if isinstance(m, nn.Conv2d)]
hooks = []
for i, (name, layer) in enumerate(conv_layers[:4]):  # 4 premières couches Conv
    hooks.append(layer.register_forward_hook(make_hook(f'conv_{i+1}')))

# Passe avant pour collecter les activations
with torch.no_grad():
    _ = model_augm(img_tensor)

# Nettoyer les hooks
for h in hooks:
    h.remove()

# Afficher 8 feature maps par couche
layer_names = list(activations_by_layer.keys())
n_layers = len(layer_names)
n_maps   = 8

fig, axes = plt.subplots(n_layers, n_maps + 1, figsize=(16, 3 * n_layers))

for row, lname in enumerate(layer_names):
    act = activations_by_layer[lname].squeeze(0).cpu()  # (C, H, W)

    # Colonne titre
    axes[row, 0].text(0.5, 0.5, f'{lname}\n{act.shape[1]}×{act.shape[2]}',
                      ha='center', va='center', fontsize=10, fontweight='bold',
                      transform=axes[row, 0].transAxes)
    axes[row, 0].axis('off')

    for col in range(n_maps):
        if col < act.shape[0]:
            axes[row, col + 1].imshow(act[col].numpy(), cmap='plasma')
        axes[row, col + 1].axis('off')
        axes[row, col + 1].set_title(f'F{col+1}', fontsize=7)

plt.suptitle('Feature Maps à différentes profondeurs — couches simples → complexes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Grad-CAM simplifié — Carte de chaleur de l'attention du modèle ────────────
# Grad-CAM utilise les gradients de la dernière couche convolutive
# pour montrer QUELLES RÉGIONS de l'image ont influencé la décision.

def simple_gradcam(model, img_tensor, target_class):
    """
    Version simplifiée de Grad-CAM.
    Retourne une heatmap de la même taille que l'image.
    """
    model.eval()
    features, grads = [], []

    # Hook sur la dernière couche Conv
    last_conv = [m for m in model.modules() if isinstance(m, nn.Conv2d)][-1]
    fh = last_conv.register_forward_hook(lambda m, i, o: features.append(o))
    bh = last_conv.register_backward_hook(lambda m, gi, go: grads.append(go[0]))

    img_tensor.requires_grad_(True)
    out = model(img_tensor)
    model.zero_grad()
    out[0, target_class].backward()

    fh.remove(); bh.remove()

    # Pondération des feature maps par les gradients
    grad_weights = grads[0].mean(dim=(2, 3), keepdim=True)  # GAP des gradients
    cam = (grad_weights * features[0]).sum(dim=1, keepdim=True)
    cam = F.relu(cam)  # Garder seulement les activations positives
    cam = F.interpolate(cam, size=(32, 32), mode='bilinear', align_corners=False)
    cam = cam.squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam


# Afficher Grad-CAM sur 4 exemples
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
indices = [42, 100, 200, 300]

for col, idx in enumerate(indices):
    img_t, label = test_cifar[idx]
    inp = img_t.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model_augm(inp).argmax(1).item()

    cam = simple_gradcam(model_augm, inp.clone(), pred)
    img_display = denorm(img_t).permute(1, 2, 0).numpy()

    # Image originale
    axes[0, col * 2].imshow(img_display)
    c = 'green' if pred == label else 'red'
    axes[0, col * 2].set_title(f'Réel: {CIFAR_CLASSES[label]}', fontsize=8, color='black')
    axes[0, col * 2].axis('off')

    # Overlay Grad-CAM
    axes[0, col * 2 + 1].imshow(img_display)
    axes[0, col * 2 + 1].imshow(cam, cmap='jet', alpha=0.5)
    axes[0, col * 2 + 1].set_title(f'Prédit: {CIFAR_CLASSES[pred]}', fontsize=8, color=c)
    axes[0, col * 2 + 1].axis('off')

for ax in axes[1]:
    ax.axis('off')

plt.suptitle('Grad-CAM — Régions ayant influencé la décision du CNN (rouge = plus important)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Analyse — Visualisation des filtres CNN

**Couche 1 — Features primitives** : les filtres apprennent des détecteurs de bords dans différentes orientations (horizontal, vertical, diagonal), des gradients de couleur, et des fréquences spatiales simples. C'est cohérent avec ce qu'on sait de la vision biologique : les cellules simples du cortex visuel V1 sont également des détecteurs d'orientations.

**Couches profondes — Features abstraites** : les feature maps des couches 3 et 4 sont plus petites (plus poolées) et détectent des patterns complexes — combinaisons de formes, textures répétitives, parties d'objets. Ces représentations sont difficilement interprétables à l'œil, ce qui illustre pourquoi le deep learning est souvent qualifié de « boîte noire ».

**Grad-CAM** révèle que le CNN focus sur les régions réellement discriminantes (ex : la tête d'un chat, la carlingue d'un avion) plutôt que le fond. Cela confirme que le modèle a appris des patterns sémantiques et non des artefacts du dataset.

Ces visualisations sont essentielles pour la **confiance dans les modèles** deployés en production : dans un contexte médical, un radiologue voudra vérifier que le CNN regarde bien la lésion et non un artéfact de l'image.

---
# 🎓 Conclusion — Récapitulatif des 4 exercices

| Exercice | Technique | Résultat clé |
|----------|-----------|-------------|
| **1 - Objets** | Data Augmentation | +X% accuracy, meilleure généralisation |
| **2 - Peu de données** | Dropout + Early Stopping + Augmentation combinés | Réduction significative de l'écart train/test |
| **3 - Transfer Learning** | ResNet18 pré-entraîné | Convergence 3× plus rapide, meilleure accuracy finale |
| **4 - Filtres** | Visualisation + Grad-CAM | Confirmation que le CNN apprend des features pertinentes |

### 🚀 Pour aller plus loin :
- **Exercice 1** : Tester d'autres architectures (EfficientNet, ViT)
- **Exercice 2** : Ajouter la régularisation L2 (weight decay) et comparer
- **Exercice 3** : Tester ResNet50 ou EfficientNet-B0 pour encore plus de performance
- **Exercice 4** : Explorer LIME ou SHAP pour une interprétabilité plus poussée